In [1]:
# Impot libraries
import pandas as pd

In [3]:
df = pd.read_csv('WRMD_VA_2016-2025_Records.csv')

In [5]:
df.columns

Index(['patients.address_found', 'patients.admitted_at',
       'patients.admitted_by', 'patients.city_found',
       'patients.clinical_signs', 'patients.common_name',
       'patients.county_found', 'patients.diagnosis', 'patients.disposition',
       'patients.found_at', 'patients.keywords', 'patients.lat_found',
       'patients.lng_found', 'patients.name', 'patients.notes_about_rescue',
       'patients.reason_for_disposition', 'patients.reasons_for_admission',
       'patients.release_type', 'species.genus', 'species.species'],
      dtype='object')

In [7]:
df.shape

(25264, 20)

## Drop Rows where there is no geolocation information

In [8]:
df[['patients.address_found', 'patients.county_found', 'patients.city_found']]

,patients.address_found,patients.county_found,patients.city_found
0,"Halfway Rd, approx 1 mile south of Fauquier Co...",Fauquier,Middleburg
1,10378 Wheatley School Road,Fauquier County,Marshall
2,Bakerton Rd one mile from 340,Jefferson County,Harper's Ferry
3,436 E Piccadilly St,NaN,Winchester
4,19549 Manor Dr,Culpeper County,Culpeper
...,...,...,...
25259,1011 Everett Ct.,Spotsylvania,Fredericksburg
25260,16879 Oliver St.,Loudoun,Paeonian Springs
25261,122 Mt. Vernon Ct.,Orange,Locust Grove
25262,2330 Kaetzel Rd.,Washington,Knoxville


In [17]:
#Profile the address_found column
df['patients.address_found'].value_counts()

patients.address_found
ss                                            294
00                                            129
unknown                                       124
X                                             120
x                                              91
                                             ... 
1030 Lake View Dr.                              1
118 Alpine Dr.                                  1
intersection of Kings Hwy and Tinsbloom Ln      1
10032 Boreland Ct                               1
19274 Harlow Sq                                 1
Name: count, Length: 17751, dtype: int64

Some of the data in 'patients.address_found' is not useful for programatically determining lat long coordinates. We want to drop the rows where the address_found starts with a alphabetic character AND either lat_found or lng_found is null

In [19]:
# How many rows have the 'patients.address_found' column starting with a non-numeric character
df[df['patients.address_found'].str[0].str.isnumeric() == False].shape

(6026, 20)

In [24]:
# Of the rows that have the 'patients.address_found' column starting with a non-numeric character, how many have a null for eithe rlat_found or long_found
df_address_profiling = df[df['patients.address_found'].str[0].str.isnumeric() == False]
df_address_profiling[['patients.lat_found', 'patients.lng_found']].isnull().sum()

patients.lat_found    3994
patients.lng_found    3994
dtype: int64

In [25]:
# determine which rows have the 'patients.address_found' column starting with a non-numeric character 
# df[df['patients.address_found'].str[0].str.isnumeric() == True]
# Set the 'patients.address_found' column to null for these rows
df.loc[df['patients.address_found'].str[0].str.isnumeric() == False, 'patients.address_found'] = None

In [34]:
# Find rows where the 'patients.address_found' column contains only numbers
df[df['patients.address_found'].str.isnumeric() == True]

,patients.address_found,patients.admitted_at,patients.admitted_by,patients.city_found,patients.clinical_signs,patients.common_name,patients.county_found,patients.diagnosis,patients.disposition,patients.found_at,patients.keywords,patients.lat_found,patients.lng_found,patients.name,patients.notes_about_rescue,patients.reason_for_disposition,patients.reasons_for_admission,patients.release_type,species.genus,species.species
1503,00,2016-09-29 18:14:00,00,00,NaN,Woodland Box Turtle,NaN,T- Unknown trauma,Released,2016-12-23,NaN,37.522251,-78.668194,NaN,NaN,ready for release,00,NaN,Terrapene,Carolina
1504,00,2016-09-29 18:14:00,00,00,NaN,Northern Mockingbird,NaN,T - Unknown trauma,Euthanized +24hr,2016-12-23,NaN,37.522251,-78.668194,NaN,NaN,NaN,00,NaN,Mimus,polyglottos
1505,00,2016-09-29 18:15:00,00,00,NaN,House Sparrow,NaN,X – Invasive,Dead on arrival,2016-12-23,NaN,37.522251,-78.668194,NaN,NaN,NaN,00,NaN,Passer,domesticus
1506,00,2016-09-30 18:15:00,00,00,NaN,Northern Raccoon,NaN,T - Unknown trauma,Released,2016-12-23,NaN,37.522251,-78.668194,NaN,NaN,NaN,00,NaN,Procyon,lotor
1507,00,2016-09-30 18:15:00,00,00,NaN,House Sparrow,NaN,X – Invasive,Euthanized in 24hr,2016-12-23,NaN,37.522251,-78.668194,NaN,NaN,NaN,00,NaN,Passer,domesticus
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19542,522,2023-07-02 14:31:00,AE,Winchester,NaN,Virginia Opossum,Frederick,B - Baby,Euthanized in 24hr,2023-07-01,"B - Baby, small, finder negligence",NaN,NaN,NaN,NaN,Poor Prognosis,Mom HBV,NaN,Didelphis,virginiana
20388,340,2023-08-28 11:45:00,AE,Waterloo,NaN,Virginia Opossum,Clarke,M - Human non-intentional,Euthanized in 24hr,2023-05-10,finder negligence,NaN,NaN,NaN,NaN,Rabies Testing,Friend of finder caught mom VAOP in a live tra...,NaN,Didelphis,virginiana
20389,340,2023-08-28 11:45:00,AE,Waterloo,NaN,Virginia Opossum,Clarke,M - Human non-intentional,Euthanized in 24hr,2023-05-10,finder negligence,NaN,NaN,NaN,NaN,Rabies Testing,Friend of finder caught mom VAOP in a live tra...,NaN,Didelphis,virginiana
22490,4297,2024-05-19 12:15:00,KS,Warrenton,NaN,Eastern Painted Turtle,Fauquier,H - Hit by Vehicle,Euthanized in 24hr,2024-05-19,NaN,NaN,NaN,NaN,NaN,Poor Prognosis,"Found on side of road, HBV",NaN,Chrysemys,Picta


In [35]:
# For rows where the 'patients.address_found' column contains only numbers, set the 'patients.address_found' column to null
df.loc[df['patients.address_found'].str.isnumeric() == True, 'patients.address_found'] = None

In [36]:
# Drop rows where all the following columns are empty: 'patients.address_found', 'patients.lat_found', 'patients.long_found'
df = df.dropna(subset=['patients.address_found', 'patients.lat_found', 'patients.lng_found'], how='all')

In [37]:
df.shape

(21225, 20)

## Convert Address to Geolocation
For rows where we have an address but no lat or long info

In [8]:
df[['patients.address_found', 'patients.lat_found', 'patients.lng_found']]

,patients.address_found,patients.lat_found,patients.lng_found
0,1179 SR-712,NaN,NaN
1,3250 Bust Head Rd,NaN,NaN
2,Intersection of Greenway and Rt 7,NaN,NaN
3,109 Keverne Ct.,NaN,NaN
4,21024 Fox Hollow Ln.,NaN,NaN
...,...,...,...
3312,225 Sister Chipmunk Ln,NaN,NaN
3313,20333 Tanager Place,NaN,NaN
3314,Found on a walking trail behind Dockside Terra...,39.040332,-77.354295
3315,297 Cedar Mtn. Ln.,NaN,NaN


In [ ]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut
import time
from concurrent.futures import ThreadPoolExecutor

# Initialize geolocator with a custom timeout value (in seconds)
geolocator = Nominatim(user_agent="Geopy Library")

# Cache to store geocoded addresses (address as key, (lat, lng) as value) to avoid repeating 
cache = {}

# Function to geocode an address 
def geocode_address(address, retries=3, delay=2):
    """
    Geocode an address with retry logic and timeout handling.
    Includes caching to avoid re-geocoding the same address.
    
    param address: The address to geocode.
    param retries: Number of retries on timeout (default 3).
    param delay: Delay (in seconds) between retries (default 2).
    return: Tuple (latitude, longitude) or (None, None) if not found.
    """
    if address in cache:
        return cache[address]  # Return cached result
    
    for attempt in range(retries):
        try:
            # Attempt to geocode with a 5-second timeout
            location = geolocator.geocode(address, timeout=5)
            
            if location:
                # Cache the result for future use
                cache[address] = (location.latitude, location.longitude)
                return location.latitude, location.longitude
            else:
                return None, None
        
        except GeocoderTimedOut:
            time.sleep(delay)  # Sleep before retrying
        except Exception as e:
            time.sleep(delay)  # Sleep before retrying
    
    return None, None

# Function to process each row in the DataFrame and update coordinates
def process_row(index, row):
    address = row['patients.address_found']
    if pd.notna(address):  # Only geocode if address is not NaN
        lat, lng = geocode_address(address)
        return index, lat, lng
    return index, None, None

# Function to populate missing latitude and longitude in the DataFrame using multi-threading
def populate_missing_coordinates(df):
    with ThreadPoolExecutor(max_workers=10) as executor:
        results = list(executor.map(lambda row: process_row(*row), df.iterrows()))
        
    for index, lat, lng in results:
        if lat is not None and lng is not None:
            df.at[index, 'patients.lat_found'] = lat
            df.at[index, 'patients.lng_found'] = lng
    return df

# Call the function to update missing latitude and longitude
df = populate_missing_coordinates(df)

# Output the updated DataFrame
df


## Determine Which ones are Vehicle Collision Related

* The patients.keyword column contains relevant information. Search for "HBV" (Hit by Vehicle).
* The patients.diagnosis column also has relevant data - "HBV" and "HBV" as a string in long from text
* the patient.reasons_for_admission column has relevent info: "HBV" among others


### Keyword Column

In [7]:
# Find all rows in the patient.keywords column that contain the string "HBV" and examine the results
df[df['patients.keywords'].str.contains('HBV', regex=False, case=False, na=False)]['patients.keywords'].value_counts()

patients.keywords
suspect HBV                                                152
B - Baby, mother killed, mother HBV, orphan                 20
B - Baby, orphan, mother HBV                                18
B - Baby, mother HBV, small                                  9
suspect HBV, mother killed, B - Baby, small                  8
B - Baby, suspect HBV                                        7
mother HBV, mother killed, orphan, B - Baby, small           6
B - Baby, small, mother HBV, orphan                          5
suspect HBV, B - Baby                                        4
suspect HBV, finder negligence                               2
trauma, suspect HBV                                          2
mother HBV, B - Baby, orphan                                 2
B - Baby, mother killed, mother HBV, orphan, dog attack      2
suspect HBV, lead                                            2
suspect HBV,  suspect cat attack                             1
suspect HBV, URI                     

In [8]:
# Grab the indexes of the rows that contain the string "HBV" in the patient.keywords column
keyword_column_hbv = df[df['patients.keywords'].str.contains('HBV', regex=False, case=False, na=False)].index

### Diagnosis Column

In [9]:
df['patients.diagnosis'].value_counts()

patients.diagnosis
T - Unknown Trauma                       768
D - Domestic animal attack               440
B - Baby                                 411
S - Non-infectious Disease               243
H - Hit by Vehicle                       166
M - Human non-intentional                158
X - Invasive                             157
I - Infectious Disease                   123
A - Abduction                            119
W - Window Strike                         83
U - Unknown                               34
K - Toxicity                              25
P - Predator Attack                       22
F - Monofilament line or fishing hook     15
J - Intentional Human Attack               7
N - Natural Event                          5
D - Domestic animal attack, baby           1
Name: count, dtype: int64

In [10]:
# Find all rows in the patient.diagnosis column that are equal to "H - Hit by Vehicle" and examine the results
df[df['patients.diagnosis'].str.contains('Vehicle', regex=False, case=False, na=False)]['patients.diagnosis'].value_counts()

patients.diagnosis
H - Hit by Vehicle    166
Name: count, dtype: int64

In [11]:
# Grab the indexes of the rows that contain the string "Vehicle" in the patient.diagnosis column
diagnosis_column_vehicle = df[df['patients.diagnosis'].str.contains('Vehicle', regex=False, case=False, na=False)].index


### Reasons for Admission Column

In [12]:
df['patients.reasons_for_admission'].value_counts()

patients.reasons_for_admission
Cat attack                                          85
HBV                                                 50
Dog attack                                          37
Abandoned                                           34
Orphaned                                            31
                                                    ..
Hit the barn wall, unable to fly after 2 hours       1
Eyes swollen and crusted shut                        1
found down on side in yard, suspected leg injury     1
Injured leg, confirmed dog had cardinal in mouth     1
hit by car, DOA                                      1
Name: count, Length: 1812, dtype: int64

In [13]:
# Set up a regex pattern to match the string "vehicle" or "HBV" or "collision" in the patient.reasons_for_admission column
vehicle_collision_regex = "vehicle|HBV|collision"
# Find rows where the patients.ressons_for_admission column contains the string "vehicle" or "HBV" or "collision" and examine the results
df[df['patients.reasons_for_admission'].str.contains(vehicle_collision_regex, regex=True, case=False, na=False)]

,admissions.case_year,admissions.hash,admissions.id,exams.age,exams.age_unit,exams.attitude,exams.bcs,exams.body,exams.cardiopulmonary,exams.cns,...,people.notes,people.organization,people.phone,people.postal_code,people.subdivision,species.class,species.family,species.genus,species.order,species.species
6,2024,NaN,7,NaN,Adult,Obtunded,Reasonable,"OD susp ruptured, fluid and blood crusted over...",NaN,NaN,...,NaN,VCA,7.037512e+09,22314,VA,Mammalia,Sciuridae,Sciurus,Rodentia,carolinensis
10,2024,NaN,11,NaN,Adult,Obtunded,Good,lower jaw crushing injury and severely swollen...,NaN,NaN,...,NaN,NaN,4.349610e+09,22601,VA,Mammalia,Sciuridae,Sciurus,Rodentia,carolinensis
24,2024,NaN,25,NaN,Adult,Alert,Good,NaN,NaN,NaN,...,NaN,NaN,5.408126e+09,22640,VA,Aves,Cathartidae,Cathartes,Cathartiformes,aura
27,2024,NaN,28,NaN,Adult,Stuporous,NaN,very weak,extremely poor jugular fill,NaN,...,NaN,FCSO Deputy C. Amari,4.439550e+09,22630,VA,Aves,Strigidae,Bubo,Strigiformes,virginianus
31,2024,NaN,32,NaN,Adult,Stuporous,Thin,See Feathers/Fur/Skin,NaN,"Mentally inappropriate, appeared to be twitchi...",...,NaN,NaN,5.712748e+09,20180,VA,Mammalia,Procyonidae,Procyon,Carnivora,lotor
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3272,2024,NaN,3271,NaN,Adult,Obtunded,Good,NaN,NaN,NaN,...,"Per HJ, finder (Richard) initially found BAOW ...",Loudoun Valley Raptors (Heather Jeweler),2.024121e+09,20165,VA,Aves,Strigidae,Strix,Strigiformes,varia
3274,2024,NaN,3273,NaN,Adult,Stuporous,Reasonable,"profuse bleeding from oral cavity, regurgitate...",NaN,NaN,...,NaN,NaN,5.408775e+09,20184,VA,Reptilia,Emydidae,Terrapene,Testudines,Carolina
3277,2024,NaN,3276,NaN,Adult,Obtunded,Good,OS large older appearing retinal tear along ve...,NaN,NaN,...,NaN,Loudoun Valley Raptor Center,7.034004e+09,NaN,VA,Aves,Strigidae,Megascops,Strigiformes,asio
3303,2024,NaN,3302,NaN,Juvenile,Depressed,Reasonable,NaN,"Open mouth breathing, moderatly dyspneic, wors...",NaN,...,NaN,NaN,5.406213e+09,22514,VA,Mammalia,Didelphidae,Didelphis,Didelphimorphia,virginiana


In [14]:
# Grab the indexes of the rows that contain the string "Vehicle" in the patient.diagnosis column
reasonforadmission_column_vehicle = df[df['patients.reasons_for_admission'].str.contains(vehicle_collision_regex, regex=True, case=False, na=False)].index

### Unify hit by vechicle indexes

In [15]:
# Unify the indexes of the rows that we suspect are related to vehicle collisions
unified_indexes = keyword_column_hbv.union(diagnosis_column_vehicle).union(reasonforadmission_column_vehicle)

In [16]:
# Create a new DataFrame with only the rows that we suspect are related to vehicle collisions
df_vehicle_collisions = df.loc[unified_indexes]

In [17]:
df_vehicle_collisions.shape

(420, 95)

### Examine disposition lat and long columns

In [18]:
df_examime_location = df[['patients.disposition_lat',
       'patients.disposition_lng', 'patients.disposition_location',
       'patients.disposition_subdivision', 'patients.dispositioned_at',
       'patients.dispositioned_by', 'patients.found_at', 'patients.keywords',
       'patients.lat_found', 'patients.lng_found']]

In [22]:
# select rows from df_examiem_location_cols where patients.disposition_lat is not null
df_examime_location[df_examime_location['patients.disposition_lat'].notnull()]

,patients.disposition_lat,patients.disposition_lng,patients.disposition_location,patients.disposition_subdivision,patients.dispositioned_at,patients.dispositioned_by,patients.found_at,patients.keywords,patients.lat_found,patients.lng_found
56,31.390460,-92.669310,Boyce,VA,2024-01-23,JR,2024-01-22,mycotoxins,NaN,NaN
57,31.390460,-92.669310,Boyce,VA,2024-01-23,JR,2024-01-22,mycotoxins,NaN,NaN
58,31.390460,-92.669310,Boyce,VA,2024-01-23,JR,2024-01-22,mycotoxins,NaN,NaN
59,31.390460,-92.669310,Boyce,VA,2024-01-23,JR,2024-01-22,mycotoxins,NaN,NaN
60,31.390460,-92.669310,Boyce,VA,2024-01-23,JR,2024-01-22,mycotoxins,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
3073,39.048614,-78.060132,Boyce,VA,2024-09-01,JA,2024-08-30,B - Baby,NaN,NaN
3088,39.048614,-78.060132,Boyce,VA,2024-09-01,JA,2024-08-31,"B - Baby, fell from nest, orphan, renesting no...",NaN,NaN
3173,39.048614,-78.060132,Boyce,VA,NaN,NaN,2024-09-05,"B - Baby, suspect cat attack",NaN,NaN
3220,39.048614,-78.060132,Boyce,VA,2024-09-16,JA,2024-09-09,"B - Baby, suspect cat attack",NaN,NaN


Examining the disposition_lat and disposition_lng columns often gives the coordination of wildlife center or animal hospitals. Suspect that this column is not meaningful for our goal of identifying hot spots of animal-vehicle conflict